# Fully Autonomous Mode

In fully autonomous mode, the agent receives a goal, plans how to achieve it, executes all necessary actions, handles failures through retry and replanning, and delivers a completion report. Humans receive results; they do not receive approval requests during execution.

This is not a default starting point. It is earned autonomy - deployed only after the agent has demonstrated reliability at lower levels (Supervised → Semi-Autonomous → Fully Autonomous).

```
Human provides goal
    │
    ▼
Agent executes agentic loop:
    │
    ├── LLM plans next action
    ├── Tool call attempted
    │       ├── Success → continue
    │       └── Failure → retry with backoff → report error to LLM → replan
    │
    ├── Self-monitoring check before each iteration
    │       ├── Within limits → continue
    │       └── Limit exceeded → escalate immediately
    │
    ▼
Task complete → Informational report delivered to human
```

**Autonomy level: 4 - End-to-end autonomous**
Escalation triggers in fully autonomous mode are exceptions, not checkpoints: retry limit exhausted, self-monitoring limit exceeded, safety boundary detected, or goal too ambiguous to plan.

In [2]:
import os
import time
import random
from dataclasses import dataclass, field
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool

### Initialize the language model

In [3]:
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, api_key=os.getenv('OPENAI_API_KEY'))
print('LLM ready:', llm.model_name)

LLM ready: gpt-4o-mini


## Self-monitoring: the safety layer for full autonomy
In supervised and semi-autonomous modes, the human provides a natural circuit-breaker. In fully autonomous mode, the agent is its own safety layer. Without explicit human checkpoints, there is nothing external to stop a runaway loop that calls expensive tools indefinitely or runs far longer than the task warrants.

`AutonomousAgentMonitor` enforces hard limits on three dimensions: total tool calls, elapsed wall-clock time, and estimated cost. The monitor is checked at the start of every loop iteration. If any limit is exceeded, execution stops immediately and the agent escalates - it does not try to finish the current step or complete a partial result.

`ExecutionRecord` is the companion data structure: one record per tool call, capturing the tool name, arguments, result, number of attempts required, and whether it succeeded. Together the monitor and the execution log form the audit trail that compensates for the absence of human checkpoints.

In [4]:
@dataclass
class ExecutionRecord:
    """A single entry in the agent's execution log.

    Attributes:
        tool: Tool name that was called.
        args: Arguments passed to the tool.
        result: Tool output or error message.
        attempt: Which attempt number this was (1 = first try, 2+ = retried).
        success: Whether the tool call succeeded.
        timestamp: Unix timestamp of the call.
    """
    tool:      str
    args:      dict
    result:    str
    attempt:   int  = 1
    success:   bool = True
    timestamp: float = field(default_factory=time.time)


class AutonomousAgentMonitor:
    """Enforce hard execution limits during fully autonomous operation.

    Checked before every loop iteration. A single exceeded limit is enough
    to trigger immediate escalation — no partial completions on overrun.
    """

    def __init__(
        self,
        max_tool_calls:      int   = 30,
        max_duration_seconds: int  = 300,
        max_cost_usd:         float = 2.0,
    ):
        """
        Args:
            max_tool_calls: Maximum tool calls before forced escalation.
            max_duration_seconds: Maximum wall-clock seconds for the task.
            max_cost_usd: Estimated spend ceiling (rough token-cost estimate).
        """
        self.max_tool_calls       = max_tool_calls
        self.max_duration_seconds = max_duration_seconds
        self.max_cost_usd         = max_cost_usd
        self.start_time           = time.time()
        self.tool_call_count      = 0
        self.estimated_cost_usd   = 0.0

    def record_tool_call(self, estimated_cost: float = 0.001) -> None:
        """Increment the tool call counter and accumulate cost estimate."""
        self.tool_call_count    += 1
        self.estimated_cost_usd += estimated_cost

    def check(self) -> tuple[bool, str]:
        """Check whether execution is within acceptable parameters.

        Returns:
            (within_limits, reason). If within_limits is False the agent
            must stop and escalate immediately rather than continuing.
        """
        elapsed = time.time() - self.start_time

        if self.tool_call_count >= self.max_tool_calls:
            return False, f'Tool call limit: {self.tool_call_count}/{self.max_tool_calls}'

        if elapsed >= self.max_duration_seconds:
            return False, f'Duration limit: {elapsed:.0f}s/{self.max_duration_seconds}s'

        if self.estimated_cost_usd >= self.max_cost_usd:
            return False, f'Cost limit: ${self.estimated_cost_usd:.3f}/${self.max_cost_usd:.2f}'

        return True, 'Within limits'

    def status(self) -> str:
        """Return a compact status string for logging."""
        elapsed = time.time() - self.start_time
        return (
            f'tools={self.tool_call_count}/{self.max_tool_calls} | '
            f'time={elapsed:.0f}s/{self.max_duration_seconds}s | '
            f'cost=${self.estimated_cost_usd:.3f}/${self.max_cost_usd:.2f}'
        )


# Demonstrate limit detection: create a monitor with a low cap and exhaust it
demo_monitor = AutonomousAgentMonitor(max_tool_calls=3, max_duration_seconds=60, max_cost_usd=0.50)
print('Initial check:', demo_monitor.check())
for _ in range(3):
    demo_monitor.record_tool_call()
print('After 3 calls:', demo_monitor.check())
print('Status line  :', demo_monitor.status())

Initial check: (True, 'Within limits')
After 3 calls: (False, 'Tool call limit: 3/3')
Status line  : tools=3/3 | time=0s/60s | cost=$0.003/$0.50


The monitor is stateful: each call to `record_tool_call` increments the counter and accumulates cost. `check()` is side-effect-free - it reads the current state and returns a verdict. This separation means the agent loop can call `check()` as many times as needed without affecting the counters.

The three limits serve different failure modes. The tool-call limit catches infinite loops (the most common failure in poorly-prompted agents). The duration limit catches tasks that are taking far longer than expected, which often indicates a planning failure. The cost limit is the financial guardrail: it prevents a single runaway task from consuming a disproportionate share of the API budget.

## Mock tools: a nightly operations audit
These tools simulate an API health audit: checking system status, fetching performance metrics, analysing error logs, running diagnostics, and saving a report. The `run_diagnostic` tool is made deliberately unreliable - a 40% failure rate per call - to demonstrate the retry logic that fully autonomous mode requires. Real production tools fail intermittently for many reasons (network timeouts, rate limits, transient service errors), and the agent must handle these without escalating to a human on every first failure.

In [5]:
random.seed(42)  # fixed seed for reproducible output in this demo


@tool
def check_service_health(service: str) -> str:
    """Check the health status of a named service.

    Args:
        service: Service name (e.g. 'api-gateway', 'auth-service').

    Returns:
        Health status report.
    """
    statuses = {
        'api-gateway':   'HEALTHY | uptime: 99.97% | last restart: 14 days ago',
        'auth-service':  'HEALTHY | uptime: 99.99% | last restart: 30 days ago',
        'data-pipeline': 'DEGRADED | uptime: 98.1% | last restart: 2 hours ago',
    }
    return statuses.get(service, f'UNKNOWN service: {service}')


@tool
def get_performance_metrics(service: str, window_minutes: int = 60) -> str:
    """Fetch performance metrics for a service over a time window.

    Args:
        service: Service name to query.
        window_minutes: Lookback window in minutes.

    Returns:
        Key performance metrics as a formatted string.
    """
    metrics = {
        'api-gateway':   f'p50:45ms p95:120ms p99:340ms error_rate:0.12% RPS:2840 (over {window_minutes}m)',
        'auth-service':  f'p50:12ms p95:38ms  p99:95ms  error_rate:0.01% RPS:890  (over {window_minutes}m)',
        'data-pipeline': f'p50:980ms p95:3.2s p99:8.1s  error_rate:2.40% RPS:42   (over {window_minutes}m)',
    }
    return metrics.get(service, f'No metrics found for: {service}')


@tool
def analyze_error_logs(service: str, limit: int = 10) -> str:
    """Retrieve and summarise recent error log entries for a service.

    Args:
        service: Service name to query.
        limit: Maximum number of recent entries to return.

    Returns:
        Error log summary string.
    """
    logs = {
        'api-gateway':   'Top errors (last 1h): TimeoutError x12, ConnectionReset x3. No new types.',
        'auth-service':  'Top errors (last 1h): TokenExpiry x2. All within normal thresholds.',
        'data-pipeline': 'Top errors (last 1h): DBConnectionPool exhausted x47, SlowQueryWarning x89. INVESTIGATE.',
    }
    return logs.get(service, f'No log data for: {service}')


@tool
def run_diagnostic(service: str) -> str:
    """Run a comprehensive diagnostic check against a service.

    Simulates a 40% failure rate to demonstrate the agent's retry logic.
    In a real deployment, failures represent network timeouts or service unavailability.

    Args:
        service: Service name to diagnose.

    Returns:
        Diagnostic report.
    """
    # 40% of calls raise an exception — the retry logic must handle these
    if random.random() < 0.4:
        raise RuntimeError(f'Diagnostic for {service} timed out — network unreachable')

    return (
        f'Diagnostic complete for {service}: '
        f'CPU 34% | Memory 61% | Disk 48% | DB connections 42/100 | Queue depth 0'
    )


@tool
def save_report(filename: str, content: str) -> str:
    """Save a report to disk. Idempotent — safe to overwrite on re-runs.

    Args:
        filename: Target filename.
        content: Report content to write.

    Returns:
        Save confirmation.
    """
    return f'[MOCK] Report saved → {filename} ({len(content)} chars)'


ALL_TOOLS = [check_service_health, get_performance_metrics, analyze_error_logs, run_diagnostic, save_report]
TOOL_MAP  = {t.name: t for t in ALL_TOOLS}

print('Tools:', [t.name for t in ALL_TOOLS])
print('Note: run_diagnostic has a simulated 40% failure rate to exercise retry logic.')

Tools: ['check_service_health', 'get_performance_metrics', 'analyze_error_logs', 'run_diagnostic', 'save_report']
Note: run_diagnostic has a simulated 40% failure rate to exercise retry logic.


## Retry logic with exponential backoff
Transient failures are a normal operating condition in production systems. A fully autonomous agent must recover from them independently - escalating to a human on every first tool failure would eliminate most of the benefit of full autonomy. The standard approach is exponential backoff: retry after 1 second, then 2 seconds, then 4 seconds. This gives transient conditions (a brief network blip, a rate-limit window) time to resolve before giving up.

`execute_with_retry` is a standalone function that wraps a single tool call with this retry logic. Making it standalone, rather than a method on the agent class, means it can be tested in isolation and its behaviour verified independently of the full agent loop.

In [6]:
def execute_with_retry(
    tool_name:   str,
    tool_args:   dict,
    tool_map:    dict,
    max_retries: int = 3,
) -> tuple[str, bool, int]:
    """Execute a tool call with exponential backoff retry on failure.

    Args:
        tool_name: Name of the tool to call.
        tool_args: Arguments to pass to the tool.
        tool_map: Dict mapping tool name to callable tool.
        max_retries: Maximum attempts before giving up (default 3).

    Returns:
        (result_str, success, attempts_made) — the result or error message,
        whether it ultimately succeeded, and how many attempts were needed.
    """
    for attempt in range(1, max_retries + 1):
        try:
            result = tool_map[tool_name].invoke(tool_args)
            # Successful call — return immediately without further retries
            return str(result), True, attempt
        except Exception as e:
            if attempt < max_retries:
                # Exponential backoff: 1s after attempt 1, 2s after attempt 2, 4s after 3...
                delay = 2 ** (attempt - 1)
                print(f'  [RETRY] {tool_name} failed (attempt {attempt}). Retrying in {delay}s...')
                time.sleep(delay)
            else:
                # All retries exhausted — return error string so the LLM can replan
                error_msg = (
                    f'Tool "{tool_name}" failed after {attempt} attempts. '
                    f'Last error: {e}. Try an alternative approach.'
                )
                return error_msg, False, attempt


# Demonstrate retry behaviour using run_diagnostic (40% failure rate).
# Reset the random seed so the demo is reproducible.
random.seed(0)
result_str, succeeded, attempts_used = execute_with_retry(
    'run_diagnostic',
    {'service': 'api-gateway'},
    TOOL_MAP,
)
print(f'Result  : {result_str[:80]}')
print(f'Success : {succeeded}')
print(f'Attempts: {attempts_used}')

Result  : Diagnostic complete for api-gateway: CPU 34% | Memory 61% | Disk 48% | DB connec
Success : True
Attempts: 1


The retry function never escalates on its own - it exhausts its retry budget and returns an error string. The error string is then fed back to the LLM as a tool result, and the model can choose a different approach. This separation of concerns - retry logic in `execute_with_retry`, replanning in the LLM - keeps each piece focused on one responsibility.

The delay schedule (1s, 2s, 4s) covers most transient network failures. For longer-lived issues like rate-limit windows, the delay multiplier or maximum retries can be increased. The `time.sleep` calls make this unsuitable for concurrent execution; in a production async agent, `asyncio.sleep` would be the right substitute.

## System prompt and the autonomous agent loop
The system prompt for fully autonomous mode does two distinctive things: it instructs the agent to plan explicitly before acting, and it specifies a structured completion report format. Both are compensating for the absence of human checkpoints. The plan makes the agent's reasoning transparent in the execution log; the structured report ensures the human receives actionable results, not a wall of status messages.

The agent function itself is the standard agentic loop with three additions: self-monitoring before each iteration, `execute_with_retry` replacing direct tool calls, and an escalation path that fires when either the monitor or the iteration cap triggers.

In [7]:
FULLY_AUTO_SYSTEM = """You are an autonomous DevOps agent in Fully Autonomous Mode.

Work through the goal methodically:
1. State a numbered PLAN before taking any action (prefix each line with PLAN:).
2. Execute each step using available tools — no permission requests, no pauses.
3. If a tool fails, you will receive the error message back; try a different approach.
4. When all steps are complete, produce a final report using exactly this format:

AUTONOMOUS EXECUTION REPORT
────────────────────────────
Goal: <original goal>
Status: COMPLETED | PARTIALLY COMPLETED
Actions taken: <brief bullet list of steps performed>
Key findings: <important results or anomalies discovered>
Saved artifacts: <files or records created, if any>
Recommendations: <suggested follow-up actions>
────────────────────────────
"""

# Bind all tools to the LLM for fully autonomous operation
llm_auto = llm.bind_tools(ALL_TOOLS)


def run_autonomous_agent(
    goal:    str,
    monitor: AutonomousAgentMonitor,
) -> dict:
    """Execute a goal fully autonomously with retry logic and self-monitoring.

    The agent plans, executes tools, handles failures, and delivers a report.
    No human interaction occurs during execution. Escalation fires only if
    the monitor detects a resource limit or the iteration cap is reached.

    Args:
        goal:    High-level goal for the agent to accomplish.
        monitor: Enforces hard limits on tool calls, duration, and cost.

    Returns:
        Result dict with status, report, tool call count, and execution log.
    """
    messages       = [SystemMessage(content=FULLY_AUTO_SYSTEM), HumanMessage(content=goal)]
    execution_log  = []

    print(f'[GOAL]   {goal}')
    print(f'[LIMITS] {monitor.status()}\n')

    for iteration in range(25):  # hard iteration cap as a final safety net
        # ── Self-monitoring check before every LLM call ──────────────────
        ok, reason = monitor.check()
        if not ok:
            # Resource limit reached — stop and report immediately
            print(f'\n[ESCALATION] Limit exceeded: {reason}')
            return {
                'status':          'escalated',
                'reason':          reason,
                'goal':            goal,
                'tool_calls_made': monitor.tool_call_count,
                'execution_log':   execution_log,
            }

        response = llm_auto.invoke(messages)
        messages.append(response)

        # No tool calls means the agent has finished and produced its report
        if not response.tool_calls:
            # Print planning lines distinctly from general agent output
            for line in response.content.splitlines():
                tag = '  [PLAN]  ' if line.startswith('PLAN:') else '  [AGENT] '
                print(f'{tag}{line}')
            print(f'\n[COMPLETE] {monitor.status()}')
            return {
                'status':          'completed',
                'goal':            goal,
                'report':          response.content,
                'tool_calls_made': monitor.tool_call_count,
                'execution_log':   execution_log,
            }

        # Log any planning text the model produced alongside tool calls
        if response.content:
            for line in response.content.splitlines():
                tag = '  [PLAN]  ' if line.startswith('PLAN:') else '  [AGENT] '
                print(f'{tag}{line}')

        # Execute each requested tool call, retrying on failure
        for tc in response.tool_calls:
            result, success, attempts = execute_with_retry(
                tc['name'], tc['args'], TOOL_MAP
            )
            monitor.record_tool_call()  # count toward the limit regardless of success

            execution_log.append(ExecutionRecord(
                tool    = tc['name'],
                args    = tc['args'],
                result  = result,
                attempt = attempts,
                success = success,
            ))

            status_tag = '[TOOL]      ' if success else '[TOOL-ERROR]'
            print(f'  {status_tag} {tc["name"]} (attempt {attempts}) → {result[:80]}')
            messages.append(ToolMessage(content=result, tool_call_id=tc['id']))

    # Fell through the iteration cap — report what was completed
    return {
        'status':          'escalated',
        'reason':          'Iteration limit reached without a completion report',
        'goal':            goal,
        'tool_calls_made': monitor.tool_call_count,
        'execution_log':   execution_log,
    }

The loop structure is familiar: call the LLM, branch on whether tool calls were requested, execute tools, feed results back. The fully autonomous additions layer on top without disrupting the core flow: the monitor check inserts cleanly before the LLM call; `execute_with_retry` replaces a direct tool invocation; the execution log appends one record per tool call.

The planning output - lines starting with `PLAN:` - is printed with a distinct tag so it is visually separable from tool execution output. This makes the execution trace readable: you can follow the agent's reasoning through the plan, then watch the tools execute step by step.

## Demonstration: nightly API health audit
The agent receives a single goal and runs to completion without human interaction. Watch for `[RETRY]` lines - these show the agent handling `run_diagnostic`'s intermittent failures autonomously. After exhausting retries, the error message is fed back to the LLM, which can try the diagnostic for a different service or acknowledge the failure in its report.

In [8]:
random.seed(42)  # reset the seed for a reproducible demo

result = run_autonomous_agent(
    goal=(
        'Run the nightly health audit for the api-gateway, auth-service, and data-pipeline. '
        'For each service: check health status, fetch 60-minute performance metrics, '
        'analyse error logs, and run a diagnostic. '
        'Save the complete audit report to "nightly_audit_2026-03-27.txt".'
    ),
    monitor=AutonomousAgentMonitor(
        max_tool_calls=25,
        max_duration_seconds=300,
        max_cost_usd=2.0,
    ),
)

[GOAL]   Run the nightly health audit for the api-gateway, auth-service, and data-pipeline. For each service: check health status, fetch 60-minute performance metrics, analyse error logs, and run a diagnostic. Save the complete audit report to "nightly_audit_2026-03-27.txt".
[LIMITS] tools=0/25 | time=0s/300s | cost=$0.000/$2.00

  [PLAN]  PLAN:
  [AGENT] 1. Check the health status of the api-gateway service.
  [AGENT] 2. Fetch 60-minute performance metrics for the api-gateway service.
  [AGENT] 3. Analyze recent error logs for the api-gateway service.
  [AGENT] 4. Run a diagnostic check on the api-gateway service.
  [AGENT] 5. Check the health status of the auth-service.
  [AGENT] 6. Fetch 60-minute performance metrics for the auth-service.
  [AGENT] 7. Analyze recent error logs for the auth-service.
  [AGENT] 8. Run a diagnostic check on the auth-service.
  [AGENT] 9. Check the health status of the data-pipeline service.
  [AGENT] 10. Fetch 60-minute performance metrics for the data-

## Inspecting the execution log
After the task completes, the execution log provides a full audit trail: which tools were called, how many attempts each required, and whether they ultimately succeeded. This log answers the question "what exactly did the agent do?" - essential for debugging unexpected results and for calibrating retry limits over time.

In [9]:
print(f'Task status   : {result["status"]}')
print(f'Tool calls    : {result["tool_calls_made"]}')
print()

# Print a summary table of every tool call and its outcome
print(f'{"#":<4} {"Tool":<35} {"Attempts":<10} {"Success"}')
print('─' * 60)
for i, record in enumerate(result['execution_log'], 1):
    # Flag any call that needed more than one attempt
    retry_note = f'  ← retried {record.attempt} times' if record.attempt > 1 else ''
    print(f'{i:<4} {record.tool:<35} {record.attempt:<10} {record.success}{retry_note}')

# Summarise retry and failure statistics
retried = [r for r in result['execution_log'] if r.attempt > 1]
failed  = [r for r in result['execution_log'] if not r.success]
print(f'\nTotal calls: {len(result["execution_log"])} | Retried: {len(retried)} | Permanently failed: {len(failed)}')
print()
print('─' * 55)
print('FINAL REPORT:')
print(result.get('report', '(no report — task escalated)'))

Task status   : completed
Tool calls    : 15

#    Tool                                Attempts   Success
────────────────────────────────────────────────────────────
1    check_service_health                1          True
2    get_performance_metrics             1          True
3    analyze_error_logs                  1          True
4    run_diagnostic                      1          True
5    check_service_health                1          True
6    get_performance_metrics             1          True
7    analyze_error_logs                  1          True
8    run_diagnostic                      3          False  ← retried 3 times
9    check_service_health                1          True
10   get_performance_metrics             1          True
11   analyze_error_logs                  1          True
12   run_diagnostic                      1          True
13   run_diagnostic                      1          True
14   analyze_error_logs                  1          True
15   save_repor

## Demonstrating resource limit escalation
Fully autonomous mode must stop itself when it hits a resource boundary. The `AutonomousAgentMonitor` provides this circuit-breaker - but only if the limits are set correctly. Below, a deliberately tight `max_tool_calls=3` forces the monitor to trigger escalation mid-task.

This demonstrates the graceful shutdown path: the agent stops at the start of the next iteration (before the next LLM call), records exactly what it completed before the limit was reached, and returns an escalation dict with full context for human review.

In [10]:
random.seed(42)

# Tight limits to force escalation mid-task and demonstrate the shutdown path
constrained_result = run_autonomous_agent(
    goal=(
        'Audit all three services (api-gateway, auth-service, data-pipeline) '
        'with health check, metrics, error logs, and diagnostic for each.'
    ),
    monitor=AutonomousAgentMonitor(
        max_tool_calls=3,        # very low — will trigger after just 3 tool calls
        max_duration_seconds=120,
        max_cost_usd=1.0,
    ),
)

print(f'\nResult status     : {constrained_result["status"]}')
print(f'Escalation reason : {constrained_result.get("reason", "N/A")}')
print(f'Tool calls before escalation: {constrained_result["tool_calls_made"]}')
print(f'\nWork completed before limit hit:')
for r in constrained_result['execution_log']:
    status = '✓' if r.success else '✗'
    print(f'  {status} {r.tool}({list(r.args.values())[0] if r.args else ""})')

[GOAL]   Audit all three services (api-gateway, auth-service, data-pipeline) with health check, metrics, error logs, and diagnostic for each.
[LIMITS] tools=0/3 | time=0s/120s | cost=$0.000/$1.00

  [PLAN]  PLAN:
  [AGENT] 1. Check the health status of the 'api-gateway' service.
  [AGENT] 2. Fetch performance metrics for the 'api-gateway' service.
  [AGENT] 3. Retrieve recent error log entries for the 'api-gateway' service.
  [AGENT] 4. Run a comprehensive diagnostic check on the 'api-gateway' service.
  [AGENT] 5. Check the health status of the 'auth-service' service.
  [AGENT] 6. Fetch performance metrics for the 'auth-service' service.
  [AGENT] 7. Retrieve recent error log entries for the 'auth-service' service.
  [AGENT] 8. Run a comprehensive diagnostic check on the 'auth-service' service.
  [AGENT] 9. Check the health status of the 'data-pipeline' service.
  [AGENT] 10. Fetch performance metrics for the 'data-pipeline' service.
  [AGENT] 11. Retrieve recent error log entries for

The escalated result carries the full execution log up to the point of the limit. A human reviewer can see exactly what was accomplished, pick up from that point manually, or adjust the resource limits and retry. This is the critical property of a well-designed escalation path: it preserves context rather than just stopping silently.

## When to use fully autonomous mode

| Prerequisite | Why it matters |
|-------------|----------------|
| Agent proven reliable at Supervised/Semi-Autonomous | Full autonomy is earned, not assumed |
| Failures are recoverable in the domain | Errors do not compound catastrophically |
| Monitoring infrastructure is in place | Agent behaviour is observable during execution |
| Rollback mechanisms exist | Errors can be corrected after the fact |
| Human review latency is a real bottleneck | The speed gain justifies the reduced oversight |

**Never use fully autonomous mode when** tasks involve irreversible actions without a verified rollback path, the domain is novel with unknown failure modes, or regulatory requirements mandate human sign-off at each step.

- **Self-monitoring is mandatory, not optional.** Without the `AutonomousAgentMonitor`, there is no circuit-breaker. The monitor enforces hard limits on tool calls, duration, and cost before each LLM call.
- **Escalation is exception-driven.** The agent only escalates when it genuinely cannot proceed - resource limits hit, iteration cap reached, or safety boundary detected. This is the structural opposite of supervised mode, where human review is the default.
- **The execution log is the audit trail.** With no human checkpoints, the log is the only record of what the agent did. Every tool call - including attempts, successes, and failures - must be recorded.
- **Full autonomy is earned progressively.** Start in Supervised Mode. Build a track record. Move to Semi-Autonomous. Build a track record. Only then enable Fully Autonomous - and start with conservative resource limits.